In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../../datasets/diabetes.csv")
df.head()

In [ ]:
df.info()

In [ ]:
# Split the data into features and target
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

#### Scaling the data

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled[:5]

In [ ]:
# train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout

In [ ]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=(X_train.shape[1])))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
model.compile(optimizer='Adam', loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
model.summary()

In [49]:
model.fit(X_train, y_train, batch_size=32, epochs=100,
          validation_data=(X_test, y_test), verbose=1)

20/20 [==============================] - 0s 3ms/step - loss: 0.4208 - accuracy: 0.8013 - val_loss: 0.5268 - val_accuracy: 0.7468
Epoch 57/100
20/20 [==============================] - 0s 2ms/step - loss: 0.4211 - accuracy: 0.7980 - val_loss: 0.5258 - val_accuracy: 0.7468
Epoch 58/100
20/20 [==============================] - 0s 2ms/step - loss: 0.4198 - accuracy: 0.8013 - val_loss: 0.5281 - val_accuracy: 0.7468
Epoch 59/100
20/20 [==============================] - 0s 2ms/step - loss: 0.4193 - accuracy: 0.7980 - val_loss: 0.5306 - val_accuracy: 0.7403
Epoch 60/100
20/20 [==============================] - 0s 3ms/step - loss: 0.4192 - accuracy: 0.7980 - val_loss: 0.5296 - val_accuracy: 0.7403
Epoch 61/100
20/20 [==============================] - 0s 3ms/step - loss: 0.4184 - accuracy: 0.7964 - val_loss: 0.5288 - val_accuracy: 0.7403
Epoch 62/100
20/20 [==============================] - 0s 4ms/step - loss: 0.4179 - accuracy: 0.7964 - val_loss: 0.5295 - val_accuracy: 0.7403
Epoch 63/100
20/20 

### Using ***keras_tuner*** to find best hyperparameters

### 1. Finding best Optimizer

In [50]:
import keras_tuner as kt

# Function to build a model for hyperparameter tuning


def build_model(hp):
    model = Sequential()
    # Add input layer with 32 neurons and ReLU activation
    model.add(Dense(32, activation='relu', input_dim=(X_train.shape[1])))
    # Add output layer with sigmoid activation for binary classification
    model.add(Dense(1, activation='sigmoid'))
    # Compile the model with a tunable optimizer and binary crossentropy loss
    model.compile(optimizer=hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta']),
                  loss='binary_crossentropy',
                  metrics=['accuracy']
                  )
    return model

In [51]:
# Initialize a RandomSearch tuner for hyperparameter tuning
tuner = kt.RandomSearch(build_model,
                        objective=kt.Objective(
                            'val_accuracy', direction='max'),
                        max_trials=5,
                        project_name='parameter/optimizer',
                        )

Reloading Tuner from .\parameter/optimizer\tuner0.json


In [52]:
tuner.search(X_train, y_train, epochs=5,
             validation_data=(X_test, y_test))

In [53]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [54]:
model = tuner.get_best_models()[0]
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 32)                288       
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 321
Trainable params: 321
Non-trainable params: 0
_________________________________________________________________


In [55]:
model.fit(X_train, y_train, batch_size=32, epochs=100,
          initial_epoch=6, validation_data=(X_test, y_test), verbose=1)

Epoch 7/100
20/20 [==============================] - 1s 8ms/step - loss: 0.5466 - accuracy: 0.7443 - val_loss: 0.5588 - val_accuracy: 0.7338
Epoch 8/100
20/20 [==============================] - 0s 3ms/step - loss: 0.5278 - accuracy: 0.7524 - val_loss: 0.5464 - val_accuracy: 0.7468
Epoch 9/100
20/20 [==============================] - 0s 3ms/step - loss: 0.5124 - accuracy: 0.7541 - val_loss: 0.5362 - val_accuracy: 0.7338
Epoch 10/100
20/20 [==============================] - 0s 3ms/step - loss: 0.4994 - accuracy: 0.7590 - val_loss: 0.5292 - val_accuracy: 0.7273
Epoch 11/100
20/20 [==============================] - 0s 2ms/step - loss: 0.4895 - accuracy: 0.7704 - val_loss: 0.5245 - val_accuracy: 0.7403
Epoch 12/100
20/20 [==============================] - 0s 3ms/step - loss: 0.4815 - accuracy: 0.7704 - val_loss: 0.5207 - val_accuracy: 0.7532
Epoch 13/100
20/20 [==============================] - 0s 4ms/step - loss: 0.4751 - accuracy: 0.7785 - val_loss: 0.5158 - val_accuracy: 0.7662
Epoch 14/

#### 2. Number of neurons in a layer

In [56]:

def build_model(hp):
    model = Sequential()
    model.add(Dense(units=hp.Int('units', min_value=8, max_value=128,
              step=8), activation='relu', input_dim=(X_train.shape[1])))

    # Add output layer with sigmoid activation for binary classification
    model.add(Dense(1, activation='sigmoid'))

    # Compile the model with a tunable optimizer and binary crossentropy loss
    model.compile(optimizer='rmsprop',
                  loss='binary_crossentropy',
                  metrics=['accuracy']
                  )
    return model

In [57]:
import keras_tuner as kt
tuner = kt.RandomSearch(
    build_model,
    objective=kt.Objective('val_accuracy', direction='max'),
    max_trials=5,
    project_name='parameter/n_neurons',
)

Reloading Tuner from .\parameter/n_neurons\tuner0.json


In [58]:
tuner.search(X_train, y_train, epochs=5,
             validation_data=(X_test, y_test))

In [59]:
tuner.get_best_hyperparameters()[0].values

{'units': 64}

#### 3. No. of Layers in the Architecture

In [60]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(72, activation='relu', input_dim=(X_train.shape[1])))

    for i in range(hp.Int('num_layers', min_value=1, max_value=8)):
        model.add(Dense(72, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='rmsprop',
                  loss='binary_crossentropy',
                  metrics=['accuracy']
                  )

    return model

In [61]:
tuner = kt.RandomSearch(
    build_model,
    objective=kt.Objective('val_accuracy', direction='max'),
    max_trials=5,
    project_name='parameter/n_layers',
)
tuner.search(X_train, y_train, epochs=5,
             validation_data=(X_test, y_test))

Reloading Tuner from .\parameter/n_layers\tuner0.json


In [62]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 2}

In [63]:
best_model = tuner.get_best_models()[0]
best_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 72)                648       
                                                                 
 dense_1 (Dense)             (None, 72)                5256      
                                                                 
 dense_2 (Dense)             (None, 72)                5256      
                                                                 
 dense_3 (Dense)             (None, 1)                 73        
                                                                 
Total params: 11,233
Trainable params: 11,233
Non-trainable params: 0
_________________________________________________________________


#### 4. Best No. of Layers and Nodes +  Best Activation + Best Optimizer

In [70]:

def build_model(hp):
    final_model = Sequential()
    counter = 0

    for i in range(hp.Int('num_layer', min_value=1, max_value=10)):
        # Condition to just add the first layer with input_dim
        # and the rest without it
        if counter == 0:
            final_model.add(
                Dense(units=hp.Int('units '+str(i), min_value=8, max_value=128, step=8),  # Number of neurons in the i-th layer
                      activation=hp.Choice('activation '+str(i), values=[
                                           'relu', 'tanh', 'sigmoid', 'leaky_relu']),  # Activation function for the i-th layer
                      # Add input_dim only for the first layer
                      input_dim=X_train.shape[1]
                      ))
            # add dropout layer
            final_model.add(Dropout(rate=hp.Float('dropout_rate' + str(i),
                                                  min_value=0.0, max_value=0.9, step=0.1)))
        # For all other layers, just add the layer without input_dim
        else:
            final_model.add(
                Dense(units=hp.Int('units '+str(i), min_value=8, max_value=128, step=8),
                      activation=hp.Choice('activation'+str(i), values=[
                                           'relu', 'tanh', 'sigmoid', 'leaky_relu']),
                      ))
            # add dropout layer
            final_model.add(Dropout(rate=hp.Float('dropout_rate' + str(i),
                                                  min_value=0.0, max_value=0.9, step=0.1)))
        counter += 1

    # Add output layer with sigmoid activation for binary classification
    final_model.add(Dense(1, activation='sigmoid'))

    final_model.compile(optimizer=hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta']),
                        loss='binary_crossentropy',
                        metrics=['accuracy']
                        )
    return final_model

In [71]:
import keras_tuner as kt
tuner = kt.RandomSearch(
    build_model,
    objective=kt.Objective('val_accuracy', direction='max'),
    max_trials=5,
    project_name='parameter/final_model'
)

In [72]:
tuner.search(X_train, y_train, epochs=5,
             validation_data=(X_test, y_test))

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.7597402334213257
Total elapsed time: 00h 00m 09s


In [73]:
tuner.get_best_hyperparameters()[0].values

{'num_layer': 1,
 'units 0': 32,
 'activation 0': 'tanh',
 'dropout_rate0': 0.2,
 'optimizer': 'rmsprop',
 'units 1': 80,
 'activation1': 'leaky_relu',
 'dropout_rate1': 0.8,
 'units 2': 32,
 'activation2': 'leaky_relu',
 'dropout_rate2': 0.0,
 'units 3': 112,
 'activation3': 'tanh',
 'dropout_rate3': 0.8,
 'units 4': 32,
 'activation4': 'relu',
 'dropout_rate4': 0.6000000000000001,
 'units 5': 48,
 'activation5': 'leaky_relu',
 'dropout_rate5': 0.8,
 'units 6': 56,
 'activation6': 'tanh',
 'dropout_rate6': 0.0,
 'units 7': 88,
 'activation7': 'sigmoid',
 'dropout_rate7': 0.0}

In [74]:
model = tuner.get_best_models()[0]
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 32)                288       
                                                                 
 dropout (Dropout)           (None, 32)                0         
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 321
Trainable params: 321
Non-trainable params: 0
_________________________________________________________________


In [75]:
model.fit(X_train, y_train, epochs=200, initial_epoch=5,
          validation_data=(X_test, y_test), verbose=1)

Epoch 6/200
20/20 [==============================] - 1s 7ms/step - loss: 0.4972 - accuracy: 0.7524 - val_loss: 0.4954 - val_accuracy: 0.7273
Epoch 7/200
20/20 [==============================] - 0s 2ms/step - loss: 0.4921 - accuracy: 0.7736 - val_loss: 0.4895 - val_accuracy: 0.7468
Epoch 8/200
20/20 [==============================] - 0s 3ms/step - loss: 0.4899 - accuracy: 0.7606 - val_loss: 0.4858 - val_accuracy: 0.7468
Epoch 9/200
20/20 [==============================] - 0s 3ms/step - loss: 0.4896 - accuracy: 0.7508 - val_loss: 0.4834 - val_accuracy: 0.7532
Epoch 10/200
20/20 [==============================] - 0s 3ms/step - loss: 0.4841 - accuracy: 0.7769 - val_loss: 0.4821 - val_accuracy: 0.7532
Epoch 11/200
20/20 [==============================] - 0s 2ms/step - loss: 0.4735 - accuracy: 0.7752 - val_loss: 0.4827 - val_accuracy: 0.7727
Epoch 12/200
20/20 [==============================] - 0s 2ms/step - loss: 0.4738 - accuracy: 0.7736 - val_loss: 0.4833 - val_accuracy: 0.7597
Epoch 13/2